In [1]:
# Install required packages
!pip install einops gradio -q

# Display GPU info
!nvidia-smi

Tue Mar 10 13:48:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

import os
import sys
import time
import json
import math
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

from torchvision import transforms
from PIL import Image

import matplotlib.pyplot as plt
from einops import rearrange, repeat

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

Using device: cuda
Number of GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [ ]:
import pickle
import gzip
from pathlib import Path


CONFIG_OPTIMIZED = {
    'img_size': 224,
    'patch_size': 16,
    'mask_ratio': 0.75,
    
    # Encoder (ViT-Base) - SAME as original assignment specification
    'encoder_embed_dim': 768,      # ViT-Base: 768d
    'encoder_depth': 12,           # ViT-Base: 12 layers
    'encoder_num_heads': 12,       # ViT-Base: 12 heads
    
    # Decoder (ViT-Small) - SAME as original assignment specification
    'decoder_embed_dim': 384,      # ViT-Small: 384d
    'decoder_depth': 12,           # ViT-Small: 12 layers
    'decoder_num_heads': 6,        # ViT-Small: 6 heads
    
    # Training - Optimized for ~3 hours on Kaggle T4 GPU
    'batch_size': 64,
    'learning_rate': 1.5e-4,
    'weight_decay': 0.05,
    'warmup_epochs': 2,
    'total_epochs': 25,            # 25 epochs in ~3 hours
    'num_workers': 4,
    'gradient_clip': 1.0,
    
    # Data optimization - Tuned for 3-hour training window
    'use_data_subset': True,
    'data_subset_ratio': 0.20,     # 20% of data → ~7 min/epoch → ~3 hours total
}

print("✅ Optimized Training Configuration:")
print(f"   Architecture: ViT-Base (B/16) encoder + ViT-Small (S/16) decoder")
print(f"   Encoder: {CONFIG_OPTIMIZED['encoder_embed_dim']}d × {CONFIG_OPTIMIZED['encoder_depth']} layers × {CONFIG_OPTIMIZED['encoder_num_heads']} heads (~86M params)")
print(f"   Decoder: {CONFIG_OPTIMIZED['decoder_embed_dim']}d × {CONFIG_OPTIMIZED['decoder_depth']} layers × {CONFIG_OPTIMIZED['decoder_num_heads']} heads (~22M params)")
print(f"   Total Parameters: ~108M")
print(f"")
print(f"   ⏱️  Training Time Optimization:")
print(f"   Epochs: {CONFIG_OPTIMIZED['total_epochs']}")
print(f"   Data: {int(CONFIG_OPTIMIZED['data_subset_ratio']*100)}% of dataset")
print(f"   Estimated time: ~7 min/epoch × 25 epochs = ~3 hours on T4 GPU")
print(f"")
print(f"   💾 HuggingFace Upload Ready:")
print(f"   Save Format: Pickle with gzip compression")
print(f"   Expected size: ~400-450 MB (suitable for HF upload)")
print(f"   Model only (no optimizer): ~430 MB → ~200-250 MB compressed")

✅ Optimized Training Configuration:
   Architecture: ViT-Base (B/16) encoder + ViT-Small (S/16) decoder
   Encoder: 768d × 12 layers × 12 heads (~86M params)
   Decoder: 384d × 12 layers × 6 heads (~22M params)
   Total Parameters: ~108M

   ⏱️  Training Time Optimization:
   Epochs: 25
   Data: 20% of dataset
   Estimated time: ~7 min/epoch × 25 epochs = ~3 hours on T4 GPU

   💾 HuggingFace Upload Ready:
   Save Format: Pickle with gzip compression
   Expected size: ~400-450 MB (suitable for HF upload)
   Model only (no optimizer): ~430 MB → ~200-250 MB compressed


In [ ]:


def save_model_pickle(model, filepath, compress=True, config=None):
 
    if isinstance(model, nn.DataParallel):
        model_state = model.module.state_dict()
    else:
        model_state = model.state_dict()
    
    save_data = {
        'model_state_dict': model_state,
        'config': config
    }
    
    if compress:
        with gzip.open(filepath, 'wb') as f:
            pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    else:
        with open(filepath, 'wb') as f:
            pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    file_size = os.path.getsize(filepath) / (1024 ** 2)
    print(f"   Saved model to: {filepath}")
    print(f"   File size: {file_size:.2f} MB")
    return file_size


def load_model_pickle(filepath, model, compress=True):
   
    if compress:
        with gzip.open(filepath, 'rb') as f:
            save_data = pickle.load(f)
    else:
        with open(filepath, 'rb') as f:
            save_data = pickle.load(f)
    
    model.load_state_dict(save_data['model_state_dict'])
    return save_data.get('config', None)


def save_for_huggingface_torch(model, save_dir, config):

    if isinstance(model, nn.DataParallel):
        model_to_save = model.module
    else:
        model_to_save = model
    

    hf_path = os.path.join(save_dir, 'mae_model_weights.pth')
    

    save_dict = {
        'model_state_dict': model_to_save.state_dict(),
        'config': {
            'img_size': config['img_size'],
            'patch_size': config['patch_size'],
            'encoder_embed_dim': config['encoder_embed_dim'],
            'encoder_depth': config['encoder_depth'],
            'encoder_num_heads': config['encoder_num_heads'],
            'decoder_embed_dim': config['decoder_embed_dim'],
            'decoder_depth': config['decoder_depth'],
            'decoder_num_heads': config['decoder_num_heads'],
            'mask_ratio': config['mask_ratio']
        }
    }
    
    # Save with PyTorch - automatically optimized
    torch.save(save_dict, hf_path, _use_new_zipfile_serialization=True)
    
    file_size = os.path.getsize(hf_path) / (1024 ** 2)
    
    print(f"\n HuggingFace Spaces Ready - PyTorch Format:")
    print(f"   File: mae_model_weights.pth")
    print(f"   Path: {hf_path}")
    print(f"   Size: {file_size:.2f} MB")
    print(f"   Format: PyTorch (.pth) - weights only")
    print(f"   Status: {' Perfect for HF Spaces (< 500MB)' if file_size < 500 else '  Large file'}")
    print(f"   Upload:  Can be directly uploaded and loaded in Gradio app")
    
    return hf_path, file_size


def load_model_from_hf_torch(filepath):
    """Load model weights from HuggingFace PyTorch format."""
    checkpoint = torch.load(filepath, map_location='cpu')
    return checkpoint['model_state_dict'], checkpoint['config']


